# Session 23 — Attention Mechanism & Transformers (Practical)

**Goal:** build every equation from today's slides with our own hands.

We will:
1. Compute attention **step by step with NumPy** (the six steps from the slides)
2. Write **scaled dot-product attention** in PyTorch — it's only 3 lines!
3. Build a full **Multi-Head Attention** module and check it against PyTorch's built-in
4. Implement and **visualize positional encoding**
5. **Peek inside a real pretrained Transformer** and plot its attention maps

> Everything here is just matrix multiplication + softmax. If you can follow Session 22's code, you can follow this.


In [ ]:
# Setup — run this first
# If anything is missing:  pip install torch numpy matplotlib transformers

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)

---
## Part 1 — Attention by Hand (NumPy)

We'll use a tiny 4-word sentence and **tiny embeddings (size 4)** so we can print every number.

Recall the six steps from the slides:

| Step | Operation |
|------|-----------|
| 1 | Create **q, k, v** for each word (embedding × learned matrices) |
| 2 | **Score**: dot product of my query with every key |
| 3 | **Scale**: divide by √d_k |
| 4 | **Softmax**: scores → weights that sum to 1 |
| 5 | Multiply each **value** by its weight |
| 6 | **Sum** the weighted values → output z |


In [ ]:
# Our toy sentence
words = ["the", "cat", "sat", "down"]

# Toy word embeddings, one row per word (in a real model these come
# from a trained embedding layer, dimension 512 — here just 4)
X = np.array([
    [1.0, 0.0, 1.0, 0.0],   # the
    [0.0, 2.0, 0.0, 2.0],   # cat
    [1.0, 1.0, 1.0, 1.0],   # sat
    [0.0, 1.0, 1.0, 0.0],   # down
])

print("X shape:", X.shape, " (4 words, embedding size 4)")

### Step 1 — Create Q, K, V

Three **learned** weight matrices project every embedding into a query, a key and a value.
Here we just pick small fixed matrices so the numbers stay readable.


In [ ]:
d_k = 3   # dimension of queries/keys/values (64 in the paper, 3 here)

# In a real model these are learned during training
W_Q = np.array([[1, 0, 1], [1, 0, 0], [0, 0, 1], [0, 1, 1]], dtype=float)
W_K = np.array([[0, 0, 1], [1, 1, 0], [0, 1, 0], [1, 1, 0]], dtype=float)
W_V = np.array([[0, 2, 0], [0, 3, 0], [1, 0, 3], [1, 1, 0]], dtype=float)

Q = X @ W_Q     # queries : "what am I looking for?"
K = X @ W_K     # keys    : "what do I contain?"
V = X @ W_V     # values  : "what do I pass on?"

print("Q =\n", Q)
print("K =\n", K)
print("V =\n", V)

### Steps 2–4 — Score, Scale, Softmax

Let's focus on **one word — "cat" (row 1)** — exactly like the slides focused on "Thinking".


In [ ]:
def softmax(x):
    # Subtracting the max is a standard trick for numerical stability
    e = np.exp(x - x.max())
    return e / e.sum()

q_cat = Q[1]                        # the query for "cat"

# Step 2: dot product of q_cat with EVERY key (including its own)
scores = K @ q_cat
print("raw scores        :", np.round(scores, 2))

# Step 3: divide by sqrt(d_k) to keep gradients stable
scores_scaled = scores / np.sqrt(d_k)
print("scaled scores     :", np.round(scores_scaled, 2))

# Step 4: softmax -> positive weights that sum to 1
weights = softmax(scores_scaled)
print("attention weights :", np.round(weights, 3))
print("sum of weights    :", weights.sum())

for w, a in zip(words, weights):
    print(f"  'cat' attends to '{w}' with weight {a:.3f}")

### Steps 5–6 — Weighted Sum of Values

Multiply each word's **value** vector by its weight and add everything up.
Words with large weights survive almost intact; the rest are drowned out.


In [ ]:
# Step 5 & 6: weighted sum of the value vectors
z_cat = weights @ V     # same as: sum(weights[i] * V[i] for i in range(4))

print("z for 'cat':", np.round(z_cat, 3))
print()
print("This vector is the new, context-aware representation of 'cat' —")
print("mostly its own value, blended with the words it attended to.")

### The whole sentence at once (matrix form)

The loop above word-by-word is exactly one matrix expression — the formula from the slides:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$


In [ ]:
def softmax_rows(m):
    e = np.exp(m - m.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

# All six steps, all four words, one line each
attn_weights = softmax_rows(Q @ K.T / np.sqrt(d_k))   # (4 words x 4 words)
Z = attn_weights @ V                                   # (4 words x d_k)

print("Attention weight matrix (each ROW sums to 1):\n", np.round(attn_weights, 3))
print()
print("Row 1 matches our hand computation for 'cat':", np.round(attn_weights[1], 3))
print()
print("Z =\n", np.round(Z, 3))

In [ ]:
# Let's SEE the attention matrix — our first attention heatmap!
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(attn_weights, cmap="Blues")
ax.set_xticks(range(4), words)
ax.set_yticks(range(4), words)
ax.set_xlabel("attends to (keys)")
ax.set_ylabel("word (queries)")
ax.set_title("Toy self-attention weights")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{attn_weights[i, j]:.2f}", ha="center", va="center",
                color="white" if attn_weights[i, j] > 0.5 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

---
## Part 2 — Scaled Dot-Product Attention in PyTorch

The exact same formula, now as a reusable PyTorch function.
Notice how short it is — **the core of every modern LLM is these 3 lines.**


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V

    Q, K, V: (..., seq_len, d_k)  — works for batches and heads too
    mask   : optional boolean matrix; True = position is BLOCKED
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / d_k ** 0.5      # 1-3: score + scale
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))  # decoder-style masking
    weights = F.softmax(scores, dim=-1)                 # 4: softmax
    return weights @ V, weights                         # 5-6: weighted sum

In [ ]:
# Sanity check: feed in our NumPy toy example — results must match
Qt = torch.tensor(Q, dtype=torch.float32)
Kt = torch.tensor(K, dtype=torch.float32)
Vt = torch.tensor(V, dtype=torch.float32)

Z_torch, W_torch = scaled_dot_product_attention(Qt, Kt, Vt)

print("PyTorch Z:\n", Z_torch.round(decimals=3))
print("\nMatches NumPy:", np.allclose(Z_torch.numpy(), Z, atol=1e-5))

### Bonus: the decoder mask

In the **decoder**, a word may only attend to itself and *earlier* words (it must not see the future).
That's one `torch.triu` away:


In [ ]:
seq_len = 4
# True above the diagonal = "blocked"
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
print("Causal mask (True = cannot attend):\n", causal_mask)

Z_masked, W_masked = scaled_dot_product_attention(Qt, Kt, Vt, mask=causal_mask)
print("\nMasked attention weights:\n", W_masked.round(decimals=3))
print("\nNote the zeros above the diagonal — no peeking at future words.")
print("This is exactly why GPT can generate text one token at a time.")

---
## Part 3 — Multi-Head Attention Module

One head can be dominated by a word attending to itself.
**Eight heads = eight different "views"** of the sentence, each with its own W_Q, W_K, W_V.

Implementation trick used by every real library: instead of 8 separate small projections,
do **one big projection** and reshape it into heads.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads     # 512 / 8 = 64, just like the paper

        # One big learned projection per role (covers all heads at once)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)   # final output projection

    def split_heads(self, x):
        # (batch, seq, d_model) -> (batch, heads, seq, d_k)
        b, s, _ = x.shape
        return x.view(b, s, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        # 1. Project and split into heads
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        # 2. Scaled dot-product attention runs on ALL heads in parallel
        heads_out, weights = scaled_dot_product_attention(Q, K, V, mask)

        # 3. Concatenate the heads back together...
        b, h, s, d = heads_out.shape
        concat = heads_out.transpose(1, 2).contiguous().view(b, s, self.d_model)

        # 4. ...and mix them with the output projection W_O
        return self.W_o(concat), weights

In [ ]:
# Shape check with paper-sized dimensions
mha = MultiHeadAttention(d_model=512, num_heads=8)

x = torch.randn(2, 10, 512)          # batch of 2 sentences, 10 words each
out, w = mha(x, x, x)                # self-attention: Q, K, V all come from x

print("input   :", tuple(x.shape))
print("output  :", tuple(out.shape), " <- same shape: plugs into the next layer")
print("weights :", tuple(w.shape),  " <- (batch, 8 heads, 10 queries, 10 keys)")

n_params = sum(p.numel() for p in mha.parameters())
print(f"parameters: {n_params:,}  (4 x 512 x 512 weight matrices + biases)")

In [ ]:
# Cross-check against PyTorch's built-in implementation (same math inside)
builtin = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)
out_b, w_b = builtin(x, x, x)

print("built-in output :", tuple(out_b.shape))
print("built-in weights:", tuple(w_b.shape), " (averaged over heads by default)")
print()
print("Different random init -> different numbers, but identical machinery.")

---
## Part 4 — Positional Encoding

Attention has **no idea about word order** — shuffle the words and you get shuffled (but otherwise identical) outputs.
The fix: **add** a position-dependent vector to each embedding before the first layer.

The paper's recipe (sines and cosines of different frequencies):

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right) \qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$


In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()          # 0, 1, 2, ... positions
    # frequency term: 10000^(2i/d) computed in log-space
    div = torch.exp(torch.arange(0, d_model, 2).float()
                    * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)    # even columns: sine
    pe[:, 1::2] = torch.cos(pos * div)    # odd  columns: cosine
    return pe

pe = positional_encoding(max_len=100, d_model=128)
print("PE shape:", tuple(pe.shape), "- one 128-dim vector per position")

In [ ]:
# The famous striped heatmap (same one as on the slides)
plt.figure(figsize=(9, 4))
plt.imshow(pe.T, cmap="RdBu", aspect="auto")
plt.xlabel("Position in sentence")
plt.ylabel("Embedding dimension")
plt.title("Sinusoidal positional encoding — every position gets a unique 'barcode'")
plt.colorbar()
plt.tight_layout()
plt.show()

# Each COLUMN is unique -> the model can tell position 3 from position 30,
# and nearby positions have similar (but not identical) patterns.

---
## Part 5 — Peek Inside a Real Transformer

Time to look at **real attention learned from real data**.
We'll load a small pretrained BERT and feed it the sentence from the slides:

> *"The animal didn't cross the street because **it** was too tired."*

Then we check: **which words does "it" attend to?**

*(First run downloads ~250 MB. If `transformers` is missing: `pip install transformers`)*


In [ ]:
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)
model.eval()

sentence = "The animal didn't cross the street because it was too tired."
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = model(**inputs)

# attentions: 12 layers, each (batch, 12 heads, seq, seq)
attentions = outputs.attentions
print("layers:", len(attentions), "| one layer's shape:", tuple(attentions[0].shape))
print("tokens:", tokens)

In [ ]:
# Where does "it" look? Average over heads in a middle layer.
it_idx = tokens.index("it")
layer = 8                                        # try different layers!
att_it = attentions[layer][0].mean(dim=0)[it_idx]   # avg over 12 heads -> row for "it"

plt.figure(figsize=(10, 3))
plt.bar(range(len(tokens)), att_it, color="#1FA8DC")
plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
plt.ylabel("attention weight")
plt.title(f'Layer {layer}: what "it" attends to (averaged over heads)')
plt.tight_layout()
plt.show()

top = att_it.topk(4)
print('"it" attends most to:',
      [(tokens[i], round(v.item(), 3)) for v, i in zip(top.values, top.indices)])

In [ ]:
# Full attention map for one layer & head — pick your favorites
layer, head = 8, 10

plt.figure(figsize=(7, 6))
plt.imshow(attentions[layer][0, head], cmap="Blues")
plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
plt.yticks(range(len(tokens)), tokens)
plt.xlabel("attends to (keys)")
plt.ylabel("word (queries)")
plt.title(f"BERT attention — layer {layer}, head {head}")
plt.colorbar()
plt.tight_layout()
plt.show()

# Explore: different (layer, head) pairs specialize in different things —
# some track the next word, some track sentence structure, some do coreference.

---
## Exercises (try before next session)

1. **Change the toy sentence.** In Part 1, edit the rows of `X` so that "down" attends mostly to "sat". Which matrix do you need to change — and why?
2. **Break the scaling.** In `scaled_dot_product_attention`, remove the `/ sqrt(d_k)` and rerun Part 5's plots with `d_k = 64`-sized random tensors. What happens to the softmax weights? *(Hint: they become extreme — one weight ≈ 1, rest ≈ 0.)*
3. **Count heads.** Modify `MultiHeadAttention` to use 4 heads of size 128 instead of 8 of size 64. Does the parameter count change?
4. **Hunt for a coreference head.** In Part 5, loop over all 12 layers × 12 heads and find the (layer, head) pair where "it" puts the most attention on "animal".

## Recap

| Concept | Where we built it |
|---------|-------------------|
| Attention = weighted average of values, weights from q·k similarity | Part 1 |
| The whole thing is one matrix formula (3 lines of code) | Part 2 |
| Causal masking = why decoders can't see the future | Part 2 |
| Multiple heads = multiple relationship "views" | Part 3 |
| Positional encoding injects word order | Part 4 |
| Real Transformers really do learn these patterns | Part 5 |

**Next session:** we stack these blocks into a full Transformer and start working with pretrained models properly.
